In [1]:
# Filtering pipeline: truncation, language, dedup, eval-overlap, QC

import json, re, hashlib
from pathlib import Path
from collections import Counter
from google.colab import drive
drive.mount('/content/drive')

PROJECT = Path('/content/drive/MyDrive/amharic_distillation')
assert PROJECT.exists(), "Run Notebooks 1-2 first!"

# Load raw teacher answers
raw = []
with open(PROJECT / 'data' / 'teacher_answers_raw.jsonl', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            obj = json.loads(line)
            if 'error' not in obj:
                raw.append(obj)
print(f"Raw records loaded: {len(raw)}")

# Pre-filter: remove any with empty responses
raw = [r for r in raw if r.get('response', '').strip()]
print(f"Non-empty records: {len(raw)}")




Mounted at /content/drive
Raw records loaded: 1500
Non-empty records: 1500


In [2]:
# Filter 1 — Truncation detection (local)
# Detect truncation locally using generation metadata:
#   hit_max_tokens=True AND eos_produced=False -> truncated

def is_truncated(record):
    return record.get('hit_max_tokens', False) and not record.get('eos_produced', True)

truncated = [r for r in raw if is_truncated(r)]
kept_after_trunc = [r for r in raw if not is_truncated(r)]
print(f"\n--- FILTER 1: Truncation ---")
print(f"Truncated (removed): {len(truncated)} ({len(truncated)/len(raw)*100:.1f}%)")
print(f"Kept: {len(kept_after_trunc)}")
for task in sorted(set(r['task'] for r in truncated)):
    ct = sum(1 for r in truncated if r['task'] == task)
    print(f"  truncated {task}: {ct}")



--- FILTER 1: Truncation ---
Truncated (removed): 6 (0.4%)
Kept: 1494
  truncated closed_book_knowledge: 5
  truncated translation: 1


In [3]:
# Filter 2
# Ge'ez ratio filter (~0.88 for Amharic tasks, inverted for am->en).
# Isolated Latin technical terms (e.g. AI, DNA) are tolerated.

def geez_ratio(s):
    """Fraction of alphabetic chars that are Ge'ez (U+1200–U+139F)."""
    letters = [c for c in s if c.isalpha()]
    if not letters:
        return 0.0
    return sum(1 for c in letters if '\u1200' <= c <= '\u139f') / len(letters)

def has_latin_runs(s, threshold=3):
    """Returns True if text has Latin word runs >= threshold consecutive words."""
    runs = re.findall(r'\b[a-zA-Z]+(?:\s+[a-zA-Z]+){' + str(threshold-1) + r',}\b', s)
    return len(runs) > 0

GEEZ_THRESHOLD_AM = 0.88  # For tasks where output should be Amharic
GEEZ_THRESHOLD_EN = 0.10  # For am->en translation, expect English

lang_rejected = []
lang_kept = []

for r in kept_after_trunc:
    text = r['response']
    gr = geez_ratio(text)
    r['geez_ratio'] = gr

    if r.get('direction') == 'am->en':
        # Expect mostly English output
        if gr > GEEZ_THRESHOLD_EN and not has_latin_runs(text, 2):
            lang_rejected.append(r)
        else:
            lang_kept.append(r)
    elif r['task'] == 'translation' and r.get('direction') == 'en->am':
        # Expect Amharic output
        if gr < GEEZ_THRESHOLD_AM and has_latin_runs(text, 3):
            lang_rejected.append(r)
        else:
            lang_kept.append(r)
    else:
        # closed_book, instruction_following, summarization -> Amharic
        if gr < GEEZ_THRESHOLD_AM and has_latin_runs(text, 3):
            lang_rejected.append(r)
        else:
            lang_kept.append(r)

print(f"\n--- FILTER 2: Language mixing ---")
print(f"Rejected: {len(lang_rejected)} ({len(lang_rejected)/len(kept_after_trunc)*100:.1f}%)")
print(f"Kept: {len(lang_kept)}")
print("\nRejected samples:")
for r in lang_rejected[:5]:
    print(f"  [{r['task']}] geez={r['geez_ratio']:.3f} | {r['response'][:80]}...")


--- FILTER 2: Language mixing ---
Rejected: 2 (0.1%)
Kept: 1492

Rejected samples:
  [instruction_following] geez=0.424 | ድንቅ ሥራ የተባለው "Things Fall Apart" ይባላል።<turn|>...
  [instruction_following] geez=0.358 | ከሁሉ የላቀ የተባለው ድርሰቱ "The Adventures of Huckleberry Finn" ይባላል።<turn|>...


In [4]:
#  Filter 3 — Exact & Near Duplicate Handling
# 1. Exact dedup primarily on prompt/content (NOT blindly on responses).
# 2. Near-duplicate compares relevant CONTENT across the ENTIRE dataset.

def norm_text(s):
    return re.sub(r'\s+', ' ', str(s).strip().lower())

def text_hash(s):
    return hashlib.md5(norm_text(s).encode('utf-8')).hexdigest()

def ngrams(s, n=3):
    words = norm_text(s).split()
    if len(words) < n:
        return set([tuple(words)]) if words else set()
    return set(tuple(words[i:i+n]) for i in range(len(words)-n+1))

def jaccard(a, b, n=3):
    sa, sb = ngrams(a, n), ngrams(b, n)
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)

def extract_candidate_content(r):
    """Extract relevant content field for task."""
    task = r.get("task", "")
    if task == "closed_book_knowledge":
        return (r.get("question") or r.get("context") or r.get("prompt", "")).strip()
    elif task == "instruction_following":
        ctx = r.get("context", "").strip()
        q = r.get("question", "").strip()
        return f"{ctx} {q}".strip() if q else ctx
    elif task in ("summarization", "translation"):
        return r.get("context", "").strip()
    return (r.get("context") or r.get("prompt", "")).strip()

# ── 3a. Exact prompt deduplication ──
prompt_hashes = {}
exact_prompt_dupes = []
unique_prompts = []

response_hashes = {}
identical_responses = []

for r in lang_kept:
    p_hash = text_hash(r["prompt"])
    r_hash = text_hash(r["response"])
    r["prompt_hash"] = p_hash
    r["response_hash"] = r_hash

    # Track duplicate responses for audit/reporting without dropping distinct prompts
    if r_hash in response_hashes:
        identical_responses.append({
            "id": r["id"],
            "matched_id": response_hashes[r_hash],
            "response": r["response"][:80],
        })
    else:
        response_hashes[r_hash] = r["id"]

    # Deduplicate primarily on prompt
    if p_hash in prompt_hashes:
        exact_prompt_dupes.append(r)
    else:
        prompt_hashes[p_hash] = r["id"]
        unique_prompts.append(r)

print(f"\n--- FILTER 3a: Exact Dedup ---")
print(f"Exact prompt duplicates removed: {len(exact_prompt_dupes)}")
print(f"Unique prompts kept: {len(unique_prompts)}")
print(f"Identical response instances observed (reported, not dropped): {len(identical_responses)}")

# ── 3b. Near-duplicate content comparison (complete dataset comparison) ──
# Compare relevant content exhaustively within each task group
by_group = {}
for r in unique_prompts:
    group_key = r['task'] if r['task'] != 'translation' else f"translation_{r.get('direction', '')}"
    by_group.setdefault(group_key, []).append(r)

JACCARD_THRESHOLD = 0.60
near_dupes = set()

print(f"\n--- FILTER 3b: Near-dedup on content (exhaustive comparison) ---")
for group_name, records in by_group.items():
    ngram_cache = [ngrams(extract_candidate_content(r), n=3) for r in records]
    group_dropped = 0
    for i in range(len(records)):
        if records[i]['id'] in near_dupes:
            continue
        sa = ngram_cache[i]
        if not sa:
            continue
        # Compare against all subsequent items in group (no i+20 restriction)
        for j in range(i + 1, len(records)):
            if records[j]['id'] in near_dupes:
                continue
            sb = ngram_cache[j]
            if not sb:
                continue
            inter = len(sa & sb)
            if inter > 0:
                sim = inter / (len(sa) + len(sb) - inter)
                if sim > JACCARD_THRESHOLD:
                    near_dupes.add(records[j]['id'])
                    group_dropped += 1
    print(f"  {group_name:25s}: {group_dropped:3d} near-duplicates removed (total {len(records)})")

deduped = [r for r in unique_prompts if r['id'] not in near_dupes]
print(f"Total near-duplicates removed: {len(near_dupes)}")
print(f"Kept after deduplication: {len(deduped)}")


--- FILTER 3a: Exact Dedup ---
Exact prompt duplicates removed: 0
Unique prompts kept: 1492
Identical response instances observed (reported, not dropped): 3

--- FILTER 3b: Near-dedup on content (exhaustive comparison) ---
  closed_book_knowledge    :  26 near-duplicates removed (total 735)
  translation_am->en       :   0 near-duplicates removed (total 1)
  translation_en->am       :   0 near-duplicates removed (total 2)
  instruction_following    : 442 near-duplicates removed (total 738)
  summarization            :   6 near-duplicates removed (total 16)
Total near-duplicates removed: 474
Kept after deduplication: 1018


In [5]:

# Filter 4 — Evaluation-Overlap Deduplication
# Compare actual CONTENT fields against held-out eval set (not shared instructions).
# Thresholds:
#   > 0.50      -> automatically remove
#   0.30 - 0.50 -> flag for manual review
#   < 0.30      -> keep

eval_path = PROJECT / 'data' / 'eval_prompts_60.csv'
if not eval_path.exists():
    from google.colab import files
    print("Please upload eval_prompts_60.csv to run evaluation deduplication:")
    uploaded = files.upload()
    for name, data in uploaded.items():
        (PROJECT / 'data' / name).write_bytes(data)

def extract_eval_content(task, prompt_str):
    """Extract content from evaluation prompt, excluding template instructions."""
    prompt_str = str(prompt_str).strip()
    if "Context:\n" in prompt_str:
        return prompt_str.split("Context:\n", 1)[1].strip()
    return prompt_str

eval_dropped = []
eval_manual_review = []
clean_from_eval = []

if eval_path.exists():
    import pandas as pd
    eval_df = pd.read_csv(eval_path, encoding='utf-8-sig')
    eval_items = []
    for _, row in eval_df.iterrows():
        eval_items.append({
            "eval_id": row['eval_id'],
            "task": row['task'],
            "direction": row.get('direction', ''),
            "content": extract_eval_content(row['task'], row['prompt']),
        })

    for r in deduped:
        cand_content = extract_candidate_content(r)
        cand_norm = norm_text(cand_content)
        cand_ng = ngrams(cand_content, n=3)

        max_sim = 0.0
        best_match = None

        for ep in eval_items:
            eval_norm = norm_text(ep["content"])
            # Exact string containment check on content
            if cand_norm == eval_norm or (len(eval_norm) > 25 and eval_norm in cand_norm):
                max_sim = 1.0
                best_match = ep
                break

            # 3-gram Jaccard on content
            eval_ng = ngrams(ep["content"], n=3)
            inter = len(cand_ng & eval_ng)
            if inter > 0:
                sim = inter / (len(cand_ng) + len(eval_ng) - inter)
                if sim > max_sim:
                    max_sim = sim
                    best_match = ep

        if max_sim > 0.50:
            eval_dropped.append({
                "record_id": r["id"],
                "task": r["task"],
                "candidate_content": cand_content[:150],
                "matched_eval_id": best_match["eval_id"],
                "matched_eval_content": best_match["content"][:150],
                "similarity": max_sim,
            })
        elif 0.30 <= max_sim <= 0.50:
            eval_manual_review.append({
                "record_id": r["id"],
                "task": r["task"],
                "candidate_content": cand_content[:150],
                "matched_eval_id": best_match["eval_id"],
                "matched_eval_content": best_match["content"][:150],
                "similarity": max_sim,
            })
            clean_from_eval.append(r)
        else:
            clean_from_eval.append(r)
else:
    clean_from_eval = deduped

print(f"\n--- FILTER 4: Evaluation-Overlap Deduplication ---")
print(f"Automatically removed (> 0.50 overlap): {len(eval_dropped)}")
print(f"Flagged for manual review (0.30 - 0.50): {len(eval_manual_review)}")
print(f"Kept (< 0.50): {len(clean_from_eval)}")

with open(PROJECT / 'data' / 'removed_eval_overlap.json', 'w', encoding='utf-8') as f:
    json.dump(eval_dropped, f, ensure_ascii=False, indent=2)

with open(PROJECT / 'data' / 'eval_manual_review.json', 'w', encoding='utf-8') as f:
    json.dump(eval_manual_review, f, ensure_ascii=False, indent=2)


--- FILTER 4: Evaluation-Overlap Deduplication ---
Automatically removed (> 0.50 overlap): 13
Flagged for manual review (0.30 - 0.50): 0
Kept (< 0.50): 1005


In [6]:
#  Filter 5 — Quality Checks (QC)
# Length thresholds, pathological repetition, unwanted commentary

MIN_RESPONSE_CHARS = 20
MAX_RESPONSE_CHARS = 3000

qc_rejected = []
qc_kept = []

for r in clean_from_eval:
    resp = r['response']
    reasons = []

    if len(resp) < MIN_RESPONSE_CHARS:
        reasons.append(f"too_short:{len(resp)}")
    if len(resp) > MAX_RESPONSE_CHARS:
        reasons.append(f"too_long:{len(resp)}")

    # Repetition: character-level loop detection
    if len(resp) > 50:
        half = resp[:len(resp)//2]
        if resp.count(half[:30]) > 2:
            reasons.append("repetitive")

    # English commentary starters on Amharic output tasks
    if r['task'] != 'translation' or r.get('direction') != 'am->en':
        bad_starts = ['Sure', 'Here', 'Note', 'I ', 'As ', 'The ', 'This ', 'My ', 'In ', 'Let ']
        if any(resp.startswith(s) for s in bad_starts):
            reasons.append("english_start")

    if reasons:
        r['qc_fail_reasons'] = reasons
        qc_rejected.append(r)
    else:
        qc_kept.append(r)

print(f"\n--- FILTER 5: Quality Checks ---")
print(f"QC rejected: {len(qc_rejected)}")
print(f"QC kept: {len(qc_kept)}")
reason_counts = Counter()
for r in qc_rejected:
    for reason in r['qc_fail_reasons']:
        reason_counts[reason.split(':')[0]] += 1
print(f"Rejection reasons: {dict(reason_counts)}")



--- FILTER 5: Quality Checks ---
QC rejected: 0
QC kept: 1005
Rejection reasons: {}


In [7]:
# Enforce Final Dataset Requirement & Task Representation
final = qc_kept

print(f"\n{'='*65}")
print("FINAL DATASET REPRESENTATION & VALIDATION")
print(f"{'='*65}")
print(f"Total examples kept: {len(final)}")

task_counts = Counter(r['task'] for r in final)
direction_counts = Counter(r.get('direction', '') for r in final if r['task'] == 'translation')

print("\nTask representation table:")
for t, c in sorted(task_counts.items()):
    print(f"  {t:25s}: {c:5d}")

print("\nTranslation direction breakdown:")
for d, c in sorted(direction_counts.items()):
    print(f"  direction {d:15s}: {c:5d}")

# ── Floor & Ceiling Assertions ──
if len(final) < 1000:
    raise RuntimeError(
        f"\n❌ DATASET FLOOR VIOLATION: Final dataset contains only {len(final)} pairs (minimum required: 1,000).\n"
        f"Do NOT weaken the filters just to reach 1,000!\n"
        f"Please return to Notebook 2 (02_teacher_generate.ipynb) and generate a top-up batch to supplement candidate data."
    )
elif len(final) > 3000:
    raise RuntimeError(
        f"\n❌ DATASET CEILING VIOLATION: Final dataset contains {len(final)} pairs (maximum allowed: 3,000)."
    )

assert 1000 <= len(final) <= 3000, f"Expected between 1000 and 3000 examples, got {len(final)}"

# Verify all required task categories remain represented
required_tasks = {"closed_book_knowledge", "instruction_following", "summarization", "translation"}
missing_tasks = required_tasks - set(task_counts.keys())
assert not missing_tasks, f"Missing required tasks in final dataset: {missing_tasks}"

# Verify both translation directions remain represented
present_directions = set(direction_counts.keys())
assert {"en->am", "am->en"}.issubset(present_directions), (
    f"Expected both 'en->am' and 'am->en' directions, got {present_directions}"
)

print("\n✅ All dataset requirements (1,000-3,000 range, tasks, directions) successfully satisfied!")

# Save final filtered dataset
with open(PROJECT / 'data' / 'dataset_filtered.json', 'w', encoding='utf-8') as f:
    json.dump(final, f, ensure_ascii=False, indent=2)

# Save rejected records with metadata for analysis
all_rejected = (
    [{"id": r["id"], "stage": "truncation", **r} for r in truncated] +
    [{"id": r["id"], "stage": "language", **r} for r in lang_rejected] +
    [{"id": r["id"], "stage": "exact_prompt_dup", **r} for r in exact_prompt_dupes] +
    [{"id": r["id"], "stage": "near_dup"} for rid in near_dupes for r in unique_prompts if r["id"] == rid] +
    [{"id": r["record_id"], "stage": "eval_overlap", **r} for r in eval_dropped] +
    [{"id": r["id"], "stage": "qc", "reasons": r.get("qc_fail_reasons"), **r} for r in qc_rejected]
)

with open(PROJECT / 'data' / 'rejected_records.json', 'w', encoding='utf-8') as f:
    json.dump(all_rejected, f, ensure_ascii=False, indent=2, default=str)

# Update LOG.md
with open(PROJECT / 'logs' / 'LOG.md', 'a', encoding='utf-8') as f:
    f.write(f"\n## Filtering Summary\n"
            f"- Raw candidates: {len(raw)}\n"
            f"- Final kept: {len(final)}\n"
            f"- Truncated dropped: -{len(truncated)}\n"
            f"- Language dropped: -{len(lang_rejected)}\n"
            f"- Exact prompt dupes: -{len(exact_prompt_dupes)}\n"
            f"- Near-content dupes: -{len(near_dupes)}\n"
            f"- Eval overlap dropped: -{len(eval_dropped)}\n"
            f"- Eval flagged for manual review: {len(eval_manual_review)}\n"
            f"- QC dropped: -{len(qc_rejected)}\n\n")




FINAL DATASET REPRESENTATION & VALIDATION
Total examples kept: 1005

Task representation table:
  closed_book_knowledge    :   709
  instruction_following    :   283
  summarization            :    10
  translation              :     3

Translation direction breakdown:
  direction am->en         :     1
  direction en->am         :     2

✅ All dataset requirements (1,000-3,000 range, tasks, directions) successfully satisfied!


In [8]:
#Convert to SFT format (Hugging Face style)

sft_records = []
for r in final:
    messages = [
        {"role": "user", "content": r['prompt']},
        {"role": "assistant", "content": r['response']},
    ]
    sft_records.append({
        "id": r['id'],
        "task": r['task'],
        "direction": r.get('direction', ''),
        "messages": messages,
    })

with open(PROJECT / 'data' / 'sft_dataset.json', 'w', encoding='utf-8') as f:
    json.dump(sft_records, f, ensure_ascii=False, indent=2)

with open(PROJECT / 'data' / 'sft_dataset.jsonl', 'w', encoding='utf-8') as f:
    for r in sft_records:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

print(f"SFT dataset: {len(sft_records)} records saved to sft_dataset.json and sft_dataset.jsonl")


SFT dataset: 1005 records saved to sft_dataset.json and sft_dataset.jsonl


In [12]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

In [13]:
#  Empirical max_seq_length determination
# Load the STUDENT tokenizer and measure actual prompt+response token lengths.

from transformers import AutoTokenizer
import numpy as np

student_tok = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")

token_lengths = []
for r in sft_records:
    full_text = r['messages'][0]['content'] + r['messages'][1]['content']
    toks = student_tok.encode(full_text, add_special_tokens=False)
    token_lengths.append(len(toks))

arr = np.array(token_lengths)
print(f"\nToken length distribution (student tokenizer):")
print(f"  Min:    {arr.min()}")
print(f"  25th:   {int(np.percentile(arr, 25))}")
print(f"  Median: {int(np.percentile(arr, 50))}")
print(f"  75th:   {int(np.percentile(arr, 75))}")
print(f"  90th:   {int(np.percentile(arr, 90))}")
print(f"  95th:   {int(np.percentile(arr, 95))}")
print(f"  99th:   {int(np.percentile(arr, 99))}")
print(f"  Max:    {arr.max()}")

p95 = int(np.percentile(arr, 95))
recommended = ((p95 // 64) + 1) * 64
print(f"\n>>> Recommended max_seq_length: {recommended}")
print(f"    (covers {sum(arr <= recommended)/len(arr)*100:.1f}% of samples at 95th percentile)")

with open(PROJECT / 'data' / 'seq_length_stats.json', 'w') as f:
    json.dump({
        "min": int(arr.min()),
        "max": int(arr.max()),
        "p50": int(np.percentile(arr, 50)),
        "p90": int(np.percentile(arr, 90)),
        "p95": p95,
        "p99": int(np.percentile(arr, 99)),
        "recommended_max_seq_length": recommended,
        "pct_covered": float(sum(arr <= recommended) / len(arr)),
    }, f, indent=2)

with open(PROJECT / 'logs' / 'LOG.md', 'a', encoding='utf-8') as f:
    f.write(f"\n## Sequence Length Analysis\n"
            f"- P95 Token Length: {p95}\n"
            f"- Recommended max_seq_length: {recommended}\n"
            f"- Dataset Coverage: {sum(arr <= recommended)/len(arr)*100:.1f}%\n\n")

print("Sequence length stats saved to seq_length_stats.json and logged.")

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]


Token length distribution (student tokenizer):
  Min:    46
  25th:   119
  Median: 146
  75th:   446
  90th:   835
  95th:   1012
  99th:   1367
  Max:    2346

>>> Recommended max_seq_length: 1024
    (covers 95.2% of samples at 95th percentile)
Sequence length stats saved to seq_length_stats.json and logged.
